# 04 - Build data source, index, indexer, and run keyword query


## Step 1 - Load environment and create Search clients


In [ ]:
import os

from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from dotenv import load_dotenv

load_dotenv("../env/.env")
subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group = os.getenv("AZURE_RESOURCE_GROUP")
search_endpoint = os.getenv("SEARCH_ENDPOINT")
storage_account = os.getenv("STORAGE_ACCOUNT_NAME")
container_name = os.getenv("STORAGE_CONTAINER_NAME", "nasa")
if not all([subscription_id, resource_group, search_endpoint, storage_account]):
    raise RuntimeError("Missing required settings in env/.env")
credential = DefaultAzureCredential()
index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)
indexer_client = SearchIndexerClient(endpoint=search_endpoint, credential=credential)
index_name = "nasa-index"
data_source_name = "nasa-datasource"
indexer_name = "nasa-indexer"


## Step 2 - Create data source using ResourceId connection string


In [ ]:
from azure.search.documents.indexes.models import SearchIndexerDataContainer, SearchIndexerDataIdentity, SearchIndexerDataSourceConnection

connection_string = (
    f"ResourceId=/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
    f"/providers/Microsoft.Storage/storageAccounts/{storage_account};"
)
data_source = SearchIndexerDataSourceConnection(
    name=data_source_name,
    type="azureblob",
    connection_string=connection_string,
    container=SearchIndexerDataContainer(name=container_name),
    identity=SearchIndexerDataIdentity(),
)
indexer_client.create_or_update_data_source_connection(data_source)
print(f"Data source ready: {data_source_name}")


## Step 3 - Create index


In [ ]:
from azure.search.documents.indexes.models import SearchField, SearchFieldDataType, SearchIndex, SimpleField

fields = [
    SimpleField(name="metadata_storage_path", type=SearchFieldDataType.String, key=True),
    SearchField(name="title", type=SearchFieldDataType.String, searchable=True),
    SearchField(name="content", type=SearchFieldDataType.String, searchable=True),
]
index = SearchIndex(name=index_name, fields=fields)
index_client.create_or_update_index(index)
print(f"Index ready: {index_name}")


## Step 4 - Create indexer and poll status


In [ ]:
import time

from azure.search.documents.indexes.models import FieldMapping, FieldMappingFunction, SearchIndexer

indexer = SearchIndexer(
    name=indexer_name,
    data_source_name=data_source_name,
    target_index_name=index_name,
    field_mappings=[
        FieldMapping(source_field_name="metadata_storage_name", target_field_name="title"),
        FieldMapping(
            source_field_name="metadata_storage_path",
            target_field_name="metadata_storage_path",
            mapping_function=FieldMappingFunction(name="base64Encode"),
        ),
    ],
)
indexer_client.create_or_update_indexer(indexer)
indexer_client.run_indexer(indexer_name)
print("Indexer started")
for attempt in range(1, 31):
    status = indexer_client.get_indexer_status(indexer_name)
    result = status.last_result
    if result and result.status in {"success", "transientFailure", "persistentFailure"}:
        print(f"Indexer status: {result.status}")
        if result.status != "success":
            raise RuntimeError(result.error_message or "Indexer failed")
        break
    print(f"Polling ({attempt}/30)")
    time.sleep(10)
else:
    raise TimeoutError("Indexer polling timed out")


## Step 5 - Run keyword query


In [ ]:
search_client = SearchClient(endpoint=search_endpoint, index_name=index_name, credential=credential)
results = list(search_client.search(search_text="argentina", top=5))
if not results:
    raise RuntimeError("No results returned for query argentina")
for idx, doc in enumerate(results, start=1):
    print(f"[{idx}] title={doc.get('title', '(no title)')}")
    print((doc.get("content") or "")[:300])
    print("-" * 80)
